# xgap: demo replay (step 2)

Thin by design -- no logic lives here. Mount Drive, clone/pull the repo,
run `setup_colab.sh`, call `scripts/run_demo_replay.py`. All control flow
lives in `xgap_code/` and `scripts/`; this notebook only sequences calls to
it.

Repo: https://github.com/AITEAM444/xgap (public)

**Run cell 1 before importing anything else, in every fresh runtime.** Colab's
`!` shell subprocess env does not propagate into this kernel's own Python
process, so `setup_colab.sh` cannot set `MUJOCO_GL` for you here -- see that
script's own comments.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

XGAP_DRIVE_ROOT = "/content/drive/MyDrive/xgap"
XGAP_REPO_URL = "https://github.com/AITEAM444/xgap.git"

## Get the code

Clones into Drive on first run, `git pull`s on every run after -- code/config
history stays on GitHub (the source of truth), Drive is just where it lives so
`setup_colab.sh` / scripts can read it and outputs can be written next to it.
No manual re-uploading of files to Drive after this point; push to GitHub and
re-run this cell instead.

In [ ]:
if os.path.isdir(f"{XGAP_DRIVE_ROOT}/.git"):
    !git -C {XGAP_DRIVE_ROOT} pull
else:
    !git clone {XGAP_REPO_URL} {XGAP_DRIVE_ROOT}

## Environment rebuild

Idempotent -- safe to re-run. This also appends `nproc` / `nvidia-smi` /
`free -g` / installed library versions to `logs/env_meta.log` (Colab hardware
varies session to session), which satisfies the "log hardware in the first
cell" requirement without duplicating that logic here.

If this prints a restart banner, use *Runtime > Restart session* and re-run
this cell once (it will no-op on the already-satisfied install step) before
continuing.

In [ ]:
!bash {XGAP_DRIVE_ROOT}/setup_colab.sh

## Smoke run first

`configs/demo_replay_smoke.yaml`: 1 suite, 1 task, 1 episode, both
`control_mode`s -- 2 episodes total. `xgap_code/harness.py` and
`xgap_code/dataset_io.py` were written against `lerobot` source without ever
being run, so the first execution is expected to surface API mismatches
(wrong kwarg, renamed attribute, etc.) -- that's plumbing work, not a science
result. See the repo README, "How to read the first real run", for how to
tell a crash apart from an actual low-success-rate finding. Do not widen the
config until this cell completes without a traceback.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke.yaml

## Full demo replay

Only run this after the smoke cell above completes cleanly (prints a
`decision` JSON, not a traceback).

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay.yaml